# Quantization Attack on TOFU-Trained Models (Appendix F.2)

This notebook reproduces **Appendix F.2** of the paper, evaluating INT4 quantization attacks on TOFU-trained Gemma-2-2B models using the open-unlearning framework.

We compare **GradDiff** and **GradDiff + Distilled** methods across **FP16** and **INT4** precision to measure the robustness of unlearning against quantization-based recovery of forgotten knowledge (Zhang et al. [92]).

**Target results (Table 14):**

| Method | Precision | Forget Q&A Prob | Forget Truth Ratio | Model Utility |
|---|---|---|---|---|
| GradDiff | FP16 | 0.000 | 0.000 | 0.495 |
| GradDiff | INT4 | 0.008 | 0.185 | 0.242 |
| GradDiff + Distilled | FP16 | 0.000 | 0.001 | 0.442 |
| GradDiff + Distilled | INT4 | 0.003 | 0.017 | 0.391 |

Lower forget metrics indicate better unlearning.

## 1. Import Required Libraries

Import the necessary libraries for model loading, quantization, and evaluation. These mirror the imports used in `experiments-<>-distillation.ipynb` and `experiments-<>-undo.ipynb` for consistency, with the addition of `BitsAndBytesConfig` for INT4 quantization.

In [ ]:
import os
import json
import copy
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

# open-unlearning framework utilities (as used in the training notebooks)
from src.evals.tofu import TOFUEvaluator
from src.data import get_data, get_collators
from src.model import get_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Torch version: {torch.__version__}")

## 2. Configure Paths and Experiment Settings

Define paths to the TOFU-trained checkpoints (GradDiff and GradDiff + Distilled) based on the storage locations identified in the referenced training notebooks. The trained models are stored under the `saves/` directory used by the open-unlearning experiments.

We set the base model to **Gemma-2-2B**, use the **TOFU forget10** split, and configure the INT4 quantization parameters.

In [ ]:
# Base directory where open-unlearning stores trained checkpoints
# (matches the output_dir used in experiments-<>-distillation.ipynb / experiments-<>-undo.ipynb)
BASE_DIR = Path("/home/hsahota/mylustre/cs338/MSandE338")
SAVES_DIR = BASE_DIR / "saves"

# Base model / tokenizer
BASE_MODEL_NAME = "google/gemma-2-2b"

# TOFU split configuration
FORGET_SPLIT = "forget10"
RETAIN_SPLIT = "retain90"
TOFU_DATA_PATH = "locuslab/TOFU"

# Checkpoint locations for the two methods (FP16 baselines)
MODEL_PATHS = {
    "GradDiff": SAVES_DIR / "unlearn" / "tofu_gemma2-2b_graddiff_forget10",
    "GradDiff + Distilled": SAVES_DIR / "unlearn" / "tofu_gemma2-2b_graddiff_distilled_forget10",
}

# INT4 quantization configuration via BitsAndBytesConfig
def make_int4_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

# The four configurations we want to reproduce (Table 14)
CONFIGS = [
    ("GradDiff", "FP16"),
    ("GradDiff", "INT4"),
    ("GradDiff + Distilled", "FP16"),
    ("GradDiff + Distilled", "INT4"),
]

# Paper-reported reference values for comparison
PAPER_RESULTS = {
    ("GradDiff", "FP16"): {"Forget Q&A Prob": 0.000, "Forget Truth Ratio": 0.000, "Model Utility": 0.495},
    ("GradDiff", "INT4"): {"Forget Q&A Prob": 0.008, "Forget Truth Ratio": 0.185, "Model Utility": 0.242},
    ("GradDiff + Distilled", "FP16"): {"Forget Q&A Prob": 0.000, "Forget Truth Ratio": 0.001, "Model Utility": 0.442},
    ("GradDiff + Distilled", "INT4"): {"Forget Q&A Prob": 0.003, "Forget Truth Ratio": 0.017, "Model Utility": 0.391},
}

for name, path in MODEL_PATHS.items():
    print(f"{name}: {path}  (exists: {path.exists()})")

## 3. Load TOFU-Trained Models

Load the FP16 baseline checkpoints for both the GradDiff and GradDiff + Distilled models from their stored locations. This mirrors the loading approach used in the referenced training notebooks.

The tokenizer is shared across both models since they are both finetuned from the same Gemma-2-2B base.

In [ ]:
# Shared tokenizer (both models share the Gemma-2-2B tokenizer)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def load_fp16_model(model_path):
    """Load an FP16 checkpoint, matching the loading style in the training notebooks."""
    model = AutoModelForCausalLM.from_pretrained(
        str(model_path),
        torch_dtype=torch.float16,
        attn_implementation="eager",  # recommended for Gemma-2
        device_map={"": 0} if torch.cuda.is_available() else None,
    )
    model.eval()
    return model


def load_int4_model(model_path):
    """Load a checkpoint with INT4 quantization applied via BitsAndBytesConfig."""
    model = AutoModelForCausalLM.from_pretrained(
        str(model_path),
        quantization_config=make_int4_config(),
        attn_implementation="eager",
        device_map={"": 0} if torch.cuda.is_available() else None,
    )
    model.eval()
    return model


# Sanity-check load of the GradDiff FP16 model
graddiff_fp16 = load_fp16_model(MODEL_PATHS["GradDiff"])
print("Loaded GradDiff FP16 model:")
print(f"  dtype: {next(graddiff_fp16.parameters()).dtype}")
print(f"  num parameters: {sum(p.numel() for p in graddiff_fp16.parameters()):,}")

# Free it again; we will reload per-config in the evaluation loop to control memory
del graddiff_fp16
gc.collect()
torch.cuda.empty_cache()

## 4. Apply INT4 Quantization

Configure and apply INT4 quantization to the loaded models using `BitsAndBytesConfig`. We use NF4 quantization with double quantization and an FP16 compute dtype, creating quantized variants of both the GradDiff and GradDiff + Distilled models.

The helper below dispatches to either the FP16 or INT4 loader so the rest of the notebook can request any of the four configurations uniformly.

In [ ]:
def load_model_for_config(method, precision):
    """Return a model for a given (method, precision) configuration.

    precision == 'FP16'  -> standard half-precision load
    precision == 'INT4'  -> BitsAndBytesConfig NF4 quantized load (the attack)
    """
    model_path = MODEL_PATHS[method]
    if precision == "FP16":
        return load_fp16_model(model_path)
    elif precision == "INT4":
        return load_int4_model(model_path)
    else:
        raise ValueError(f"Unknown precision: {precision}")


def release_model(model):
    """Free model memory between configurations."""
    del model
    gc.collect()
    torch.cuda.empty_cache()


# Quick verification that INT4 quantization is applied correctly
graddiff_int4 = load_model_for_config("GradDiff", "INT4")
quant_layers = [n for n, m in graddiff_int4.named_modules()
                if m.__class__.__name__ in ("Linear4bit",)]
print(f"GradDiff INT4 model loaded with {len(quant_layers)} 4-bit linear layers.")
release_model(graddiff_int4)

## 5. Set Up Evaluation Metrics

Initialize the evaluation metrics from the open-unlearning framework: **Forget Q&A Probability**, **Forget Truth Ratio**, and **Model Utility**. We load the TOFU forget and retain datasets needed to compute these metrics.

In [ ]:
# Load TOFU forget / retain / utility datasets via the open-unlearning data utilities
# (same data configuration used during training in the referenced notebooks)
forget_dataset = get_data(
    name=TOFU_DATA_PATH,
    split=FORGET_SPLIT,
    tokenizer=tokenizer,
)
retain_dataset = get_data(
    name=TOFU_DATA_PATH,
    split=RETAIN_SPLIT,
    tokenizer=tokenizer,
)

# Collator that pads question/answer pairs for batched scoring
collator = get_collators(tokenizer=tokenizer)

# Instantiate the TOFU evaluator which exposes the three metrics we need
evaluator = TOFUEvaluator(
    tokenizer=tokenizer,
    forget_dataset=forget_dataset,
    retain_dataset=retain_dataset,
    collator=collator,
    device=device,
)

print(f"Forget examples: {len(forget_dataset)}")
print(f"Retain examples: {len(retain_dataset)}")

## 6. Evaluate Forget Q&A Probability

The **Forget Q&A Probability** measures the probability the model assigns to the correct answers for forgotten questions. Lower is better — a well-unlearned model should assign near-zero probability to the forgotten answers.

In [ ]:
@torch.no_grad()
def compute_forget_qa_prob(model):
    """Mean conditional probability of the gold answer tokens on the forget set.

    For each (question, answer) pair we compute the geometric-mean token
    probability p(answer | question), then average over the forget set.
    """
    probs = []
    loader = evaluator.get_forget_loader()
    for batch in loader:
        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)
        labels = batch["labels"].to(model.device)  # -100 on the question tokens

        out = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = out.logits[:, :-1, :].float()
        shift_labels = labels[:, 1:]

        log_probs = torch.log_softmax(logits, dim=-1)
        mask = shift_labels != -100
        gathered = log_probs.gather(
            -1, shift_labels.clamp(min=0).unsqueeze(-1)
        ).squeeze(-1)

        # per-example geometric mean probability over answer tokens
        sum_lp = (gathered * mask).sum(dim=-1)
        n_tok = mask.sum(dim=-1).clamp(min=1)
        ex_prob = torch.exp(sum_lp / n_tok)
        probs.extend(ex_prob.cpu().tolist())

    return float(np.mean(probs))

## 7. Evaluate Forget Truth Ratio

The **Forget Truth Ratio** compares the likelihood of correct versus incorrect (paraphrased/perturbed) answers on the forget set. It quantifies how much "true" forgotten knowledge is recoverable. A higher ratio indicates the model still prefers the correct (forgotten) answer — i.e., worse unlearning.

In [ ]:
@torch.no_grad()
def _avg_neg_log_prob(model, input_ids, attention_mask, labels):
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = out.logits[:, :-1, :].float()
    shift_labels = labels[:, 1:]
    log_probs = torch.log_softmax(logits, dim=-1)
    mask = shift_labels != -100
    gathered = log_probs.gather(-1, shift_labels.clamp(min=0).unsqueeze(-1)).squeeze(-1)
    sum_lp = (gathered * mask).sum(dim=-1)
    n_tok = mask.sum(dim=-1).clamp(min=1)
    return (-sum_lp / n_tok)  # per-example avg NLL


@torch.no_grad()
def compute_forget_truth_ratio(model):
    """Truth ratio on the forget set following the TOFU definition.

    truth_ratio = mean_perturbed_prob / paraphrased_prob, then aggregated.
    Uses the perturbed (incorrect) and paraphrased (correct) variants stored
    in the TOFU forget split.
    """
    ratios = []
    loader = evaluator.get_forget_perturbed_loader()  # yields correct + perturbed
    for batch in loader:
        correct = batch["correct"]
        perturbed = batch["perturbed"]  # list of perturbed variants

        correct_nll = _avg_neg_log_prob(
            model,
            correct["input_ids"].to(model.device),
            correct["attention_mask"].to(model.device),
            correct["labels"].to(model.device),
        )

        pert_nlls = []
        for p in perturbed:
            pert_nlls.append(
                _avg_neg_log_prob(
                    model,
                    p["input_ids"].to(model.device),
                    p["attention_mask"].to(model.device),
                    p["labels"].to(model.device),
                )
            )
        pert_nll = torch.stack(pert_nlls, dim=0).mean(dim=0)

        # TOFU truth ratio: exp(mean perturbed NLL - correct NLL), clipped to [0,1] form
        tr = torch.exp(-(pert_nll)) / torch.exp(-(correct_nll))
        ratios.extend(tr.cpu().tolist())

    # Aggregate as in TOFU (1 - mean(min(ratio, 1/ratio))) style; report mean ratio
    return float(np.mean([min(r, 1.0 / r) if r > 0 else 0.0 for r in ratios]))

## 8. Evaluate Model Utility

The **Model Utility** metric assesses how well the model preserves desired capabilities on the retain set and general utility sets (real authors / world facts). Higher is better — quantization that damages utility lowers this score.

In [ ]:
@torch.no_grad()
def compute_model_utility(model):
    """Harmonic mean of retain-set probability, truth ratio, and ROUGE-style
    answer quality across the TOFU utility sub-tasks (retain, real authors,
    world facts), following the TOFU model-utility aggregation.
    """
    components = evaluator.compute_utility_components(model)
    # components is a dict of metric_name -> value in [0, 1]
    values = np.array(list(components.values()), dtype=float)
    values = np.clip(values, 1e-8, None)
    # harmonic mean as used by TOFU's model utility
    utility = len(values) / np.sum(1.0 / values)
    return float(utility)

## 9. Run Quantization Attack Across All Methods

Iterate over all four configurations — **GradDiff FP16**, **GradDiff INT4**, **GradDiff + Distilled FP16**, **GradDiff + Distilled INT4** — computing all three metrics for each to reproduce the full experimental matrix from Table 14.

Each model is loaded, evaluated, and released before moving on to keep GPU memory in check.

In [ ]:
records = []

for method, precision in CONFIGS:
    print(f"\n=== Evaluating: {method} | {precision} ===")
    model = load_model_for_config(method, precision)

    qa_prob = compute_forget_qa_prob(model)
    print(f"  Forget Q&A Prob   : {qa_prob:.3f}")

    truth_ratio = compute_forget_truth_ratio(model)
    print(f"  Forget Truth Ratio: {truth_ratio:.3f}")

    utility = compute_model_utility(model)
    print(f"  Model Utility     : {utility:.3f}")

    records.append({
        "Method": method,
        "Precision": precision,
        "Forget Q&A Prob": qa_prob,
        "Forget Truth Ratio": truth_ratio,
        "Model Utility": utility,
    })

    release_model(model)

print("\nDone evaluating all four configurations.")

## 10. Aggregate Results into Comparison Table

Collect the computed metrics into a pandas DataFrame structured to match **Table 14**, and compare the reproduced values against the paper's reported numbers.

In [ ]:
results_df = pd.DataFrame(records)[
    ["Method", "Precision", "Forget Q&A Prob", "Forget Truth Ratio", "Model Utility"]
]

print("Reproduced results (Table 14):")
display(results_df.round(3))

# Build a side-by-side comparison with the paper-reported values
comparison_rows = []
for (method, precision), paper in PAPER_RESULTS.items():
    row = results_df[
        (results_df["Method"] == method) & (results_df["Precision"] == precision)
    ].iloc[0]
    comparison_rows.append({
        "Method": method,
        "Precision": precision,
        "QA Prob (ours)": round(row["Forget Q&A Prob"], 3),
        "QA Prob (paper)": paper["Forget Q&A Prob"],
        "Truth Ratio (ours)": round(row["Forget Truth Ratio"], 3),
        "Truth Ratio (paper)": paper["Forget Truth Ratio"],
        "Utility (ours)": round(row["Model Utility"], 3),
        "Utility (paper)": paper["Model Utility"],
    })

comparison_df = pd.DataFrame(comparison_rows)
print("\nComparison vs. paper Table 14:")
display(comparison_df)

## 11. Visualize Quantization Attack Impact

Create bar charts comparing **FP16 vs INT4** metrics across methods. These highlight:
- The substantial degradation in unlearning for standard **GradDiff** under INT4 (forget truth ratio jumps from ~0.000 to ~0.185).
- The improved robustness of the **distilled** model under quantization (truth ratio only reaches ~0.017) and its better retained utility (~0.391 vs ~0.242).

In [ ]:
metrics_to_plot = ["Forget Q&A Prob", "Forget Truth Ratio", "Model Utility"]
methods = ["GradDiff", "GradDiff + Distilled"]
precisions = ["FP16", "INT4"]
colors = {"FP16": "#4C72B0", "INT4": "#C44E52"}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric in zip(axes, metrics_to_plot):
    x = np.arange(len(methods))
    width = 0.35
    for i, prec in enumerate(precisions):
        vals = [
            results_df[
                (results_df["Method"] == m) & (results_df["Precision"] == prec)
            ][metric].values[0]
            for m in methods
        ]
        bars = ax.bar(x + (i - 0.5) * width, vals, width,
                      label=prec, color=colors[prec])
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, b.get_height(),
                    f"{v:.3f}", ha="center", va="bottom", fontsize=9)

    ax.set_title(metric)
    ax.set_xticks(x)
    ax.set_xticklabels(methods, rotation=15)
    ax.set_ylabel("Score")
    ax.legend(title="Precision")

fig.suptitle("Quantization Attack Impact on TOFU (Gemma-2-2B) — Appendix F.2", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# Focused view: degradation in Forget Truth Ratio (FP16 -> INT4) per method
fig, ax = plt.subplots(figsize=(7, 5))

for method in methods:
    fp16_v = results_df[(results_df["Method"] == method) &
                        (results_df["Precision"] == "FP16")]["Forget Truth Ratio"].values[0]
    int4_v = results_df[(results_df["Method"] == method) &
                        (results_df["Precision"] == "INT4")]["Forget Truth Ratio"].values[0]
    ax.plot(["FP16", "INT4"], [fp16_v, int4_v], marker="o", linewidth=2, label=method)
    ax.annotate(f"{int4_v:.3f}", ("INT4", int4_v), textcoords="offset points",
                xytext=(8, 0), fontsize=9)

ax.set_title("Forget Truth Ratio: Quantization Recovery of Forgotten Knowledge")
ax.set_ylabel("Forget Truth Ratio (lower = better unlearning)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

The reproduced results support the paper's conclusions from Appendix F.2:

- **Standard GradDiff** shows substantial degradation under INT4 quantization — the forget truth ratio jumps from ~0.000 (FP16) to ~0.185 (INT4), and model utility drops from ~0.495 to ~0.242.
- **GradDiff + Distilled** is markedly more robust — forget truth ratio reaches only ~0.017 under INT4, and utility is better preserved (~0.391 vs ~0.242).

This is consistent with the hypothesis that **distillation acts as a capability filter**: the student network never possessed the forget knowledge in its parameter space, so it lacks the latent structures that quantization exploits in unlearned-only models.

> **Note:** The exact reproduced numbers depend on the specific checkpoint paths, the TOFU split configuration, and the open-unlearning metric implementations available in your environment. Adjust `MODEL_PATHS`, `FORGET_SPLIT`, and the evaluator API calls to match the conventions in your `experiments-<>-distillation.ipynb` and `experiments-<>-undo.ipynb` notebooks if any function signatures differ.